# Chapter 12 — Neural Networks That Write Like Shakespeare: Recurrent Layers for Variable-Length Data

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning)
**Chapter 12:** Neural Networks That Write Like Shakespeare: Recurrent Layers for Variable-Length Data

Notebook ini berisi reproduksi kode dari buku (diambil dari repository
resmi penulis) beserta penjelasan konsep dan rangkuman chapter.

## Pendahuluan

Chapter 12 membahas **recurrent neural network (RNN)**, jaringan untuk
data **berpanjang variabel** (kalimat, teks).

### Tantangan panjang sembarang

Jaringan biasa memerlukan input berukuran tetap, sedangkan kalimat
panjangnya berbeda-beda. Salah satu solusi: **rata-rata vektor kata**
(*bag of words*). Cara ini ternyata kuat, tetapi **mengabaikan urutan
kata** sehingga tidak bisa membedakan "dog bites man" dan "man bites dog".

### Mengurus urutan

Ide: gunakan **matriks transisi** untuk menggabungkan embedding kata satu
per satu secara berurutan: `sent_vector = sent_vector.dot(W) + embedding`.
Jika awalnya matriks transisi dipilih sebagai **identitas** (matriks yang
tidak mengubah apa pun), hasilnya sama dengan menjumlahkan embedding
(bag of words). Dengan melatih matriks transisi, jaringan mempelajari
cara membuat **vektor kalimat** yang berguna (urutan penting).

### Melatih RNN

Memakai dataset **bAbI** (tugas pertanyaan-jawaban sederhana seperti
"Mary pindah ke kamar mandi. Sandra pergi ke taman. Di mana Mary?").
Forward propagation dilakukan untuk panjang sembarang dan menyimpan
setiap state; backpropagation mundur melewati setiap langkah waktu, lalu
weights diperbarui.

## Persiapan Data

File data diunduh dari repository resmi buku agar notebook bisa langsung dijalankan di Colab.

In [ ]:
# Unduh file data dari repository resmi buku (iamtrask/Grokking-Deep-Learning).
# Jika unduhan gagal, unggah file secara manual lewat google.colab.files.upload().
import os, urllib.request

RAW = 'https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/'

def ambil(nama):
    folder = os.path.dirname(nama)
    if folder:
        os.makedirs(folder, exist_ok=True)
    if os.path.exists(nama):
        print('sudah ada:', nama)
        return
    try:
        urllib.request.urlretrieve(RAW + nama, nama)
        print('terunduh :', nama)
    except Exception as e:
        print('GAGAL mengunduh', nama, '->', e)
        print('Unggah file tersebut secara manual: from google.colab import files; files.upload()')

for f in ['reviews.txt', 'labels.txt', 'tasksv11/en/qa1_single-supporting-fact_train.txt']:
    ambil(f)

# Download & Preprocess the IMDB Dataset

Mengunduh dan memproses data IMDB seperti chapter 11 (kosakata dan
indeks kata), karena awal chapter ini membandingkan representasi kalimat.

In [ ]:
# Download reviews.txt and labels.txt from here: https://github.com/udacity/deep-learning/tree/master/sentiment-network

def pretty_print_review_and_label(i):
   print(labels[i] + "\t:\t" + reviews[i][:80] + "...")

g = open('reviews.txt','r') # What we know!
reviews = list(map(lambda x:x[:-1],g.readlines()))
g.close()

g = open('labels.txt','r') # What we WANT to know!
labels = list(map(lambda x:x[:-1].upper(),g.readlines()))
g.close()


# Preprocess dataset:

import sys

f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

f = open('labels.txt')
raw_labels = f.readlines()
f.close()

tokens = list(map(lambda x:set(x.split(" ")),raw_reviews))

vocab = set()
for sent in tokens:
    for word in sent:
        if(len(word)>0):
            vocab.add(word)
vocab = list(vocab)

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            ""
    input_dataset.append(list(set(sent_indices)))

target_dataset = list()
for label in raw_labels:
    if label == 'positive\n':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

# The Surprising Power of Averaged Word Vectors

**Kekuatan rata-rata vektor kata:** kalimat dapat diwakili oleh rata-rata
embedding kata-katanya; walau sederhana, hasilnya mampu menjawab banyak
hal, tetapi kehilangan informasi urutan.

In [ ]:
import numpy as np
norms = np.sum(weights_0_1 * weights_0_1,axis=1)
norms.resize(norms.shape[0],1)
normed_weights = weights_0_1 * norms

def make_sent_vect(words):
    indices = list(map(lambda x:word2index[x],filter(lambda x:x in word2index,words)))
    return np.mean(normed_weights[indices],axis=0)

reviews2vectors = list()
for review in tokens: # tokenized reviews
    reviews2vectors.append(make_sent_vect(review))
reviews2vectors = np.array(reviews2vectors)

def most_similar_reviews(review):
    v = make_sent_vect(review)
    scores = Counter()
    for i,val in enumerate(reviews2vectors.dot(v)):
        scores[i] = val
    most_similar = list()
    
    for idx,score in scores.most_common(3):
        most_similar.append(raw_reviews[idx][0:40])
    return most_similar

most_similar_reviews(['boring','awful'])

# Matrices that Change Absolutely Nothing

**Matriks identitas:** perkalian dengan matriks identitas tidak
mengubah vektor, sehingga dapat dipakai sebagai nilai awal matriks transisi.
Pada awal training RNN berperilaku seperti bag of words; seiring training,
matriks berubah dan mulai menangkap urutan.

In [ ]:
import numpy as np

a = np.array([1,2,3])
b = np.array([0.1,0.2,0.3])
c = np.array([-1,-0.5,0])
d = np.array([0,0,0])

identity = np.eye(3)
print(identity)

In [ ]:
print(a.dot(identity))
print(b.dot(identity))
print(c.dot(identity))
print(d.dot(identity))

In [ ]:
this = np.array([2,4,6])
movie = np.array([10,10,10])
rocks = np.array([1,1,1])

print(this + movie + rocks)
print((this.dot(identity) + movie).dot(identity) + rocks)

# Forward Propagation in Python

**Forward propagation:** untuk setiap kata, vektor sebelumnya dikalikan
matriks transisi lalu ditambah embedding kata saat ini. Hasil akhirnya
dimasukkan ke layer output untuk memprediksi kata berikutnya.

In [ ]:
import numpy as np

def softmax(x_):
    x = np.atleast_2d(x_)
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

word_vects = {}
word_vects['yankees'] = np.array([[0.,0.,0.]])
word_vects['bears'] = np.array([[0.,0.,0.]])
word_vects['braves'] = np.array([[0.,0.,0.]])
word_vects['red'] = np.array([[0.,0.,0.]])
word_vects['socks'] = np.array([[0.,0.,0.]])
word_vects['lose'] = np.array([[0.,0.,0.]])
word_vects['defeat'] = np.array([[0.,0.,0.]])
word_vects['beat'] = np.array([[0.,0.,0.]])
word_vects['tie'] = np.array([[0.,0.,0.]])

sent2output = np.random.rand(3,len(word_vects))

identity = np.eye(3)

In [ ]:
layer_0 = word_vects['red']
layer_1 = layer_0.dot(identity) + word_vects['socks']
layer_2 = layer_1.dot(identity) + word_vects['defeat']

pred = softmax(layer_2.dot(sent2output))
print(pred)

# How do we Backpropagate into this?

**Backpropagation melalui waktu:** karena tiap langkah bergantung pada
langkah sebelumnya, error pada output harus dipropagasikan mundur melewati
setiap langkah (*backpropagation through time*), dengan delta tiap layer
dihitung dari delta layer setelahnya.

In [ ]:
y = np.array([1,0,0,0,0,0,0,0,0]) # target one-hot vector for "yankees"

pred_delta = pred - y
layer_2_delta = pred_delta.dot(sent2output.T)
defeat_delta = layer_2_delta * 1 # can ignore the "1" like prev. chapter
layer_1_delta = layer_2_delta.dot(identity.T)
socks_delta = layer_1_delta * 1 # again... can ignore the "1"
layer_0_delta = layer_1_delta.dot(identity.T)
alpha = 0.01
word_vects['red'] -= layer_0_delta * alpha
word_vects['socks'] -= socks_delta * alpha
word_vects['defeat'] -= defeat_delta * alpha
identity -= np.outer(layer_0,layer_1_delta) * alpha
identity -= np.outer(layer_1,layer_2_delta) * alpha
sent2output -= np.outer(layer_2,pred_delta) * alpha

# Let's Train it!

**Melatih RNN:** mempersiapkan data bAbI (pertanyaan dan jawaban),
kosakata, embedding, matriks transisi, dan weights output.

In [ ]:
import sys,random,math
from collections import Counter
import numpy as np

f = open('tasksv11/en/qa1_single-supporting-fact_train.txt','r')
raw = f.readlines()
f.close()

tokens = list()
for line in raw[0:1000]:
    tokens.append(line.lower().replace("\n","").split(" ")[1:])

print(tokens[0:3])

In [ ]:
vocab = set()
for sent in tokens:
    for word in sent:
        vocab.add(word)

vocab = list(vocab)

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i
    
def words2indices(sentence):
    idx = list()
    for word in sentence:
        idx.append(word2index[word])
    return idx

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

In [ ]:
np.random.seed(1)
embed_size = 10

# word embeddings
embed = (np.random.rand(len(vocab),embed_size) - 0.5) * 0.1

# embedding -> embedding (initially the identity matrix)
recurrent = np.eye(embed_size)

# sentence embedding for empty sentence
start = np.zeros(embed_size)

# embedding -> output weights
decoder = (np.random.rand(embed_size, len(vocab)) - 0.5) * 0.1

# one hot lookups (for loss function)
one_hot = np.eye(len(vocab))

# Forward Propagation with Arbitrary Length

Forward propagation untuk **panjang sembarang**: daftar `layers` menyimpan
state tiap langkah, sehingga bisa dipakai kembali saat backpropagation.

In [ ]:
def predict(sent):
    
    layers = list()
    layer = {}
    layer['hidden'] = start
    layers.append(layer)

    loss = 0

    # forward propagate
    preds = list()
    for target_i in range(len(sent)):

        layer = {}

        # try to predict the next term
        layer['pred'] = softmax(layers[-1]['hidden'].dot(decoder))

        loss += -np.log(layer['pred'][sent[target_i]])

        # generate the next hidden state
        layer['hidden'] = layers[-1]['hidden'].dot(recurrent) + embed[sent[target_i]]
        layers.append(layer)

    return layers, loss

# Backpropagation with Arbitrary Length

Backpropagation untuk panjang sembarang: berjalan mundur dari langkah
terakhir ke langkah pertama, menghitung `layer_0_delta` dan mengumpulkan
delta untuk matriks transisi dan embedding.

In [ ]:
# forward
for iter in range(30000):
    alpha = 0.001
    sent = words2indices(tokens[iter%len(tokens)][1:])
    layers,loss = predict(sent) 

    # back propagate
    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]
        target = sent[layer_idx-1]

        if(layer_idx > 0):  # if not the first layer
            layer['output_delta'] = layer['pred'] - one_hot[target]
            new_hidden_delta = layer['output_delta'].dot(decoder.transpose())

            # if the last layer - don't pull from a later one becasue it doesn't exist
            if(layer_idx == len(layers)-1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())
        else: # if the first layer
            layer['hidden_delta'] = layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())

# Weight Update with Arbitrary Length

Pembaruan weights untuk panjang sembarang: gradient tiap langkah
dijumlahkan lalu weights diperbarui (`alpha`) sekali per contoh.

In [ ]:
# forward
for iter in range(30000):
    alpha = 0.001
    sent = words2indices(tokens[iter%len(tokens)][1:])

    layers,loss = predict(sent) 

    # back propagate
    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]
        target = sent[layer_idx-1]

        if(layer_idx > 0):
            layer['output_delta'] = layer['pred'] - one_hot[target]
            new_hidden_delta = layer['output_delta'].dot(decoder.transpose())

            # if the last layer - don't pull from a 
            # later one becasue it doesn't exist
            if(layer_idx == len(layers)-1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())
        else:
            layer['hidden_delta'] = layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())

    # update weights
    start -= layers[0]['hidden_delta'] * alpha / float(len(sent))
    for layer_idx,layer in enumerate(layers[1:]):
        
        decoder -= np.outer(layers[layer_idx]['hidden'], layer['output_delta']) * alpha / float(len(sent))
        
        embed_idx = sent[layer_idx]
        embed[embed_idx] -= layers[layer_idx]['hidden_delta'] * alpha / float(len(sent))
        recurrent -= np.outer(layers[layer_idx]['hidden'], layer['hidden_delta']) * alpha / float(len(sent))
        
    if(iter % 1000 == 0):
        print("Perplexity:" + str(np.exp(loss/len(sent))))

# Execution and Output Analysis

Menjalankan training dan menganalisis keluaran: error turun dan jaringan
mulai memprediksi kata berikutnya dengan benar. Menurut buku, RNN linear
hanya memerlukan beberapa kata terakhir dan belajar melupakan kata yang
lebih lama.

In [ ]:
sent_index = 4

l,_ = predict(words2indices(tokens[sent_index]))

print(tokens[sent_index])

for i,each_layer in enumerate(l[1:-1]):
    input = tokens[sent_index][i]
    true = tokens[sent_index][i+1]
    pred = vocab[each_layer['pred'].argmax()]
    print("Prev Input:" + input + (' ' * (12 - len(input))) +\
          "True:" + true + (" " * (15 - len(true))) + "Pred:" + pred)

## Key Ideas

- Rata-rata vektor kata sederhana tetapi mengabaikan urutan.
- Matriks transisi (awal: identitas) menggabungkan kata secara berurutan.
- RNN membuat vektor kalimat berukuran tetap dari urutan berpanjang berbeda.
- Backpropagation through time mempropagasikan error mundur melewati tiap langkah.

## Kesimpulan


Chapter 12 menunjukkan bahwa **RNN dapat memprediksi pada urutan
berpanjang sembarang** dengan membuat representasi (vektor) berukuran
tetap bagi urutan berpanjang berbeda. Kuncinya bukan hanya apa yang
diingat vektor, melainkan juga **apa yang dilupakannya**. Namun RNN
linear ini tidak memiliki nonlinearitas, yang menjadi keterbatasan dan
motivasi untuk **LSTM** pada chapter berikutnya.